In [5]:
import os
import dotenv
import cv2
import numpy as np
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature, AnalyzeResult, AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential


dotenv.load_dotenv("../common/credentials.env")

True

In [6]:
path_to_sample_documents = "../common/data/sampleinvoice.jpeg"
endpoint = os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"]
key = os.environ["DOCUMENTINTELLIGENCE_API_KEY"]

In [7]:
# sample document
document_intelligence_client = DocumentIntelligenceClient(
        endpoint=endpoint, credential=AzureKeyCredential(key)
    )

In [8]:
#formUrl = "https://github.com/Azure-Samples/document-intelligence-code-samples/blob/main/Data/invoice/simple-invoice.png?raw=true"

# Load the image
f = open(path_to_sample_documents, "rb")       

poller = document_intelligence_client.begin_analyze_document(
    "prebuilt-layout",
    body=f,
    #AnalyzeDocumentRequest(url_source=formUrl), #--if you want to use URL instead of file
    features=[DocumentAnalysisFeature.QUERY_FIELDS],    # Specify which add-on capabilities to enable.
    query_fields=["Address", "InvoiceNumber"],  # Set the features and provide a comma-separated list of field names.
)
result: AnalyzeResult = poller.result()
print("Here are extra fields in result:\n")
if result.documents:
    for doc in result.documents:
        if doc.fields and doc.fields["Address"]:
            print(f"Address: {doc.fields['Address'].value_string}")
            print(f"bounding regions: {doc.fields["Address"].bounding_regions}")
        if doc.fields and doc.fields["InvoiceNumber"]:
            print(f"Invoice number: {doc.fields['InvoiceNumber'].value_string}")
            print(f"bounding regions: {doc.fields["InvoiceNumber"].bounding_regions}")

Here are extra fields in result:

Address: 380 Francisco St, 94133 San Francisco, CA, US.
bounding regions: [{'pageNumber': 1, 'polygon': [43, 234, 449, 234, 449, 257, 43, 257]}]
Invoice number: INV09080012
bounding regions: [{'pageNumber': 1, 'polygon': [932, 102, 1049, 102, 1049, 122, 932, 122]}]


In [9]:
image = cv2.imread('..\\common\\data\\simple-invoice.png')

#address: 
#dpi = 96
#polygon_in_pixels = [[int(x * dpi), int(y * dpi)] for x, y in zip(polygon_in_inches[::2], polygon_in_inches[1::2])]
polygon_in_inches = [186, 397, 1568, 397, 1568, 519, 186, 519]
polygon_in_pixelsd = np.array([polygon_in_inches], np.int32).reshape((-1, 1, 2))
cv2.polylines(image, [polygon_in_pixelsd], isClosed=True, color=(255, 0, 0), thickness=2)

#InvoiceNumber
polygon_in_inches = [124, 797, 272, 796, 272, 830, 124, 830]
polygon_in_pixelsd = np.array([polygon_in_inches], np.int32).reshape((-1, 1, 2))
cv2.polylines(image, [polygon_in_pixelsd], isClosed=True, color=(255, 0, 0), thickness=2)

# Save the modified image
cv2.imwrite('output_image.png', image)

True